# NBA Playoff Prediction — Modeling
**Authors:** Olivia, Benjamin  
**Project:** NBA Playoff Matchup Predictor  

---
## Setup
Make sure Anthony's preprocessing notebook has been run first and the `output/` folder exists with:
- `X_train.csv`, `y_train.csv` — regular season games with team-level differential features
- `X_playoff.csv`, `y_playoff.csv` — playoff games (our real test set)
- `master_stats.csv` — per-team per-season feature table

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.neural_network import MLPClassifier
import xgboost as xgb

pd.set_option('display.max_columns', 100)
print('Libraries loaded.')

In [ ]:
# Load preprocessed data from Anthony's notebook
X_train    = pd.read_csv('output/X_train.csv')
y_train    = pd.read_csv('output/y_train.csv').squeeze()
X_playoff  = pd.read_csv('output/X_playoff.csv')
y_playoff  = pd.read_csv('output/y_playoff.csv').squeeze()
master_stats = pd.read_csv('output/master_stats.csv')

print(f'Training set:  {X_train.shape}')
print(f'Playoff set:   {X_playoff.shape}')
print(f'Master stats:  {master_stats.shape}')

---
## Step 1: Define Features

We use **differential features** (T1 minus T2) — same approach as March ML Mania. The model sees the gap between teams, not raw values, which makes predictions symmetric.

In [ ]:
# Grab all differential columns
diff_features = [c for c in X_train.columns if c.endswith('_Diff')]

print(f'Total differential features: {len(diff_features)}')
print(diff_features)

In [ ]:
# Core feature set — prioritized by expected predictive value
CORE_FEATURES = [
    'End_Season_Elo_Diff',       # ELO gap — strongest single predictor
    'Net_Rating_Diff',           # Points scored minus allowed differential
    'Overall_Win_Pct_Diff',      # Regular season win % gap
    'Last10_Win_Pct_Diff',       # Recent form / momentum
    'Away_Win_Pct_Diff',         # Road performance (matters for away playoff games)
    'Avg_PTS_For_Diff',          # Offensive output gap
    'Avg_PTS_Ag_Diff',           # Defensive output gap
    'Avg_FG_PCT_Diff',           # Shooting efficiency gap
    'Avg_FG3_PCT_Diff',          # 3-point shooting gap
    'Avg_REB_Diff',              # Rebounding gap
    'Avg_AST_Diff',              # Assist (ball movement) gap
    'Avg_Opp_FG_PCT_Diff',       # Defensive FG% allowed gap
    'Avg_Opp_FG3_PCT_Diff',      # Defensive 3P% allowed gap
]

# Filter to only features that actually exist in the dataset
CORE_FEATURES = [f for f in CORE_FEATURES if f in X_train.columns]
print(f'Using {len(CORE_FEATURES)} core features')

X_train_core   = X_train[CORE_FEATURES]
X_playoff_core = X_playoff[CORE_FEATURES]

---
## Step 2: Baseline — ELO-Only Model

Before any ML, let's see how well a pure ELO rating predicts playoff outcomes. This is our floor.

In [ ]:
def elo_win_probability(elo_diff):
    """Convert ELO difference to win probability (standard formula)."""
    return 1 / (1 + 10 ** (-elo_diff / 400))

# Apply to playoff games
elo_preds_playoff = elo_win_probability(X_playoff['End_Season_Elo_Diff'])

elo_brier  = brier_score_loss(y_playoff, elo_preds_playoff)
elo_logloss = log_loss(y_playoff, elo_preds_playoff)
elo_auc    = roc_auc_score(y_playoff, elo_preds_playoff)

print('=== ELO Baseline (Playoffs) ===')
print(f'Brier Score: {elo_brier:.5f}  (lower is better, random = 0.25)')
print(f'Log Loss:    {elo_logloss:.5f}')
print(f'AUC:         {elo_auc:.4f}   (higher is better, random = 0.5)')

---
## Step 3: Leave-One-Season-Out Validation

Walk-forward cross-validation by year — the model only ever trains on past seasons when predicting a future one. Prevents data leakage.

In [ ]:
def walk_forward_cv(X, y, seasons, model, features, label='Model'):
    """
    Leave-one-season-out walk-forward validation.
    Trains on all seasons before val_season, predicts val_season.
    """
    unique_seasons = sorted(seasons.unique())
    results = []

    for val_season in unique_seasons:
        # Only use past seasons for training (no data leakage)
        train_mask = seasons < val_season
        val_mask   = seasons == val_season

        if train_mask.sum() == 0:
            continue  # Skip first season (no training data yet)

        X_tr, y_tr = X[train_mask][features], y[train_mask]
        X_vl, y_vl = X[val_mask][features],   y[val_mask]

        model.fit(X_tr, y_tr)
        preds = model.predict_proba(X_vl)[:, 1]

        brier  = brier_score_loss(y_vl, preds)
        ll     = log_loss(y_vl, preds)
        auc    = roc_auc_score(y_vl, preds) if y_vl.nunique() > 1 else np.nan

        results.append({'Season': val_season, 'Brier': brier, 'LogLoss': ll, 'AUC': auc})
        print(f'{label} | Season {val_season} — Brier: {brier:.5f}, AUC: {auc:.4f}')

    results_df = pd.DataFrame(results)
    print(f'\n{label} Mean Brier: {results_df["Brier"].mean():.5f}')
    print(f'{label} Mean AUC:   {results_df["AUC"].mean():.4f}')
    return results_df

---
## Step 4: Model 1 — Logistic Regression

In [ ]:
logistic_model = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=1000, random_state=42)
)

seasons_train = X_train['Season']

lr_results = walk_forward_cv(
    X_train, y_train, seasons_train,
    logistic_model, CORE_FEATURES,
    label='Logistic Regression'
)

In [ ]:
# Evaluate on actual playoff games
logistic_model.fit(X_train[CORE_FEATURES], y_train)
lr_playoff_preds = logistic_model.predict_proba(X_playoff_core)[:, 1]

print('=== Logistic Regression — Playoff Performance ===')
print(f'Brier Score: {brier_score_loss(y_playoff, lr_playoff_preds):.5f}')
print(f'Log Loss:    {log_loss(y_playoff, lr_playoff_preds):.5f}')
print(f'AUC:         {roc_auc_score(y_playoff, lr_playoff_preds):.4f}')

---
## Step 5: Model 2 — XGBoost

Same architecture as March ML Mania — handles non-linear feature interactions well.

In [ ]:
xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

xgb_results = walk_forward_cv(
    X_train, y_train, seasons_train,
    xgb_model, CORE_FEATURES,
    label='XGBoost'
)

In [ ]:
# Evaluate on actual playoff games
xgb_model.fit(X_train[CORE_FEATURES], y_train)
xgb_playoff_preds = xgb_model.predict_proba(X_playoff_core)[:, 1]

print('=== XGBoost — Playoff Performance ===')
print(f'Brier Score: {brier_score_loss(y_playoff, xgb_playoff_preds):.5f}')
print(f'Log Loss:    {log_loss(y_playoff, xgb_playoff_preds):.5f}')
print(f'AUC:         {roc_auc_score(y_playoff, xgb_playoff_preds):.4f}')

---
## Step 6: Model 3 — Neural Network (MLP)

In [ ]:
mlp_model = make_pipeline(
    StandardScaler(),
    MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation='relu',
        max_iter=500,
        learning_rate_init=0.001,
        random_state=42,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=20
    )
)

mlp_results = walk_forward_cv(
    X_train, y_train, seasons_train,
    mlp_model, CORE_FEATURES,
    label='Neural Network'
)

In [ ]:
# Evaluate on actual playoff games
mlp_model.fit(X_train[CORE_FEATURES], y_train)
mlp_playoff_preds = mlp_model.predict_proba(X_playoff_core)[:, 1]

print('=== Neural Network — Playoff Performance ===')
print(f'Brier Score: {brier_score_loss(y_playoff, mlp_playoff_preds):.5f}')
print(f'Log Loss:    {log_loss(y_playoff, mlp_playoff_preds):.5f}')
print(f'AUC:         {roc_auc_score(y_playoff, mlp_playoff_preds):.4f}')

---
## Step 7: Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['ELO Baseline', 'Logistic Regression', 'XGBoost', 'Neural Network'],
    'Playoff Brier': [
        brier_score_loss(y_playoff, elo_preds_playoff),
        brier_score_loss(y_playoff, lr_playoff_preds),
        brier_score_loss(y_playoff, xgb_playoff_preds),
        brier_score_loss(y_playoff, mlp_playoff_preds),
    ],
    'Playoff AUC': [
        roc_auc_score(y_playoff, elo_preds_playoff),
        roc_auc_score(y_playoff, lr_playoff_preds),
        roc_auc_score(y_playoff, xgb_playoff_preds),
        roc_auc_score(y_playoff, mlp_playoff_preds),
    ],
    'CV Brier (mean)': [
        np.nan,
        lr_results['Brier'].mean(),
        xgb_results['Brier'].mean(),
        mlp_results['Brier'].mean(),
    ]
})

comparison = comparison.sort_values('Playoff Brier')
comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Brier Score (lower = better)
axes[0].barh(comparison['Model'], comparison['Playoff Brier'], color='steelblue')
axes[0].set_xlabel('Brier Score (lower is better)')
axes[0].set_title('Playoff Brier Score by Model')
axes[0].axvline(0.25, color='red', linestyle='--', label='Random (0.25)')
axes[0].legend()

# AUC (higher = better)
axes[1].barh(comparison['Model'], comparison['Playoff AUC'], color='darkorange')
axes[1].set_xlabel('AUC (higher is better)')
axes[1].set_title('Playoff AUC by Model')
axes[1].axvline(0.5, color='red', linestyle='--', label='Random (0.5)')
axes[1].legend()

plt.tight_layout()
plt.savefig('output/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/model_comparison.png')

---
## Step 8: Feature Importance (XGBoost)

Which features actually drive playoff predictions?

In [ ]:
importances = pd.DataFrame({
    'Feature': CORE_FEATURES,
    'Importance': xgb_model.feature_importances_
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(8, 6))
plt.barh(importances['Feature'], importances['Importance'], color='steelblue')
plt.xlabel('XGBoost Feature Importance (gain)')
plt.title('What Predicts NBA Playoff Wins?')
plt.tight_layout()
plt.savefig('output/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/feature_importance.png')

---
## Step 9: Logistic Regression Coefficients

More interpretable than XGBoost — shows direction and magnitude of each feature.

In [ ]:
# Extract coefficients from the logistic regression pipeline
lr_coefs = pd.DataFrame({
    'Feature': CORE_FEATURES,
    'Coefficient': logistic_model.named_steps['logisticregression'].coef_[0]
}).sort_values('Coefficient')

colors = ['tomato' if c < 0 else 'steelblue' for c in lr_coefs['Coefficient']]

plt.figure(figsize=(8, 6))
plt.barh(lr_coefs['Feature'], lr_coefs['Coefficient'], color=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (positive = helps T1 win)')
plt.title('Logistic Regression Coefficients')
plt.tight_layout()
plt.savefig('output/lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10: Predict Current Season Playoff Matchups

Generate win probabilities for every possible playoff matchup in the most recent season.

In [ ]:
import itertools

def predict_playoff_matchups(master_stats, model, features, season):
    """
    Generates win probability predictions for all possible playoff matchups
    in a given season, using the best model.
    """
    current = master_stats[master_stats['Season'] == season].copy()
    teams = current['TeamID'].unique()

    # Generate all T1 vs T2 combinations
    combos = list(itertools.combinations(sorted(teams), 2))
    matchups = pd.DataFrame(combos, columns=['T1_TeamID', 'T2_TeamID'])
    matchups['Season'] = season

    # Merge stats for T1
    feat_cols = [c for c in current.columns if c not in ['Season', 'TeamID', 'Win_Pct']]
    matchups = matchups.merge(
        current, left_on=['Season', 'T1_TeamID'], right_on=['Season', 'TeamID'], how='left'
    ).drop(columns=['TeamID'])
    matchups = matchups.rename(columns={c: f'T1_{c}' for c in feat_cols})

    # Merge stats for T2
    matchups = matchups.merge(
        current, left_on=['Season', 'T2_TeamID'], right_on=['Season', 'TeamID'], how='left'
    ).drop(columns=['TeamID'])
    matchups = matchups.rename(columns={c: f'T2_{c}' for c in feat_cols})

    # Compute differentials
    for feat in feat_cols:
        matchups[f'{feat}_Diff'] = matchups[f'T1_{feat}'] - matchups[f'T2_{feat}']

    # Predict
    X_pred = matchups[features].fillna(0)
    matchups['T1_Win_Prob'] = model.predict_proba(X_pred)[:, 1]
    matchups['T2_Win_Prob'] = 1 - matchups['T1_Win_Prob']

    return matchups[['Season', 'T1_TeamID', 'T2_TeamID', 'T1_Win_Prob', 'T2_Win_Prob']]

# Use most recent season in master_stats
latest_season = master_stats['Season'].max()
predictions = predict_playoff_matchups(master_stats, xgb_model, CORE_FEATURES, latest_season)

# Show top matchups (closest to 50/50)
predictions['Competitiveness'] = (predictions['T1_Win_Prob'] - 0.5).abs()
print(f'Most competitive predicted matchups in {latest_season}:')
predictions.sort_values('Competitiveness').head(10)

In [ ]:
# Save predictions
predictions.to_csv(f'output/playoff_predictions_{latest_season}.csv', index=False)
print(f'Predictions saved to output/playoff_predictions_{latest_season}.csv')

---
## Step 11: Playoff-Specific Analysis

Do regular season stats translate to the playoffs? Let's check if the model's calibration changes in the postseason vs regular season.

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(7, 6))

for preds, label, color in [
    (xgb_playoff_preds, 'XGBoost',            'steelblue'),
    (lr_playoff_preds,  'Logistic Regression', 'darkorange'),
    (elo_preds_playoff, 'ELO Baseline',        'green'),
]:
    frac_pos, mean_pred = calibration_curve(y_playoff, preds, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker='o', label=label, color=color)

ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title('Calibration Curves — Playoff Games')
ax.legend()
plt.tight_layout()
plt.savefig('output/calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to output/calibration_curves.png')

In [ ]:
# Brier score by playoff season — does the model get better or worse over time?
seasons_playoff = X_playoff['Season']
brier_by_season = []

for season in sorted(seasons_playoff.unique()):
    mask = seasons_playoff == season
    brier_by_season.append({
        'Season': season,
        'XGBoost':   brier_score_loss(y_playoff[mask], xgb_playoff_preds[mask]),
        'Logistic':  brier_score_loss(y_playoff[mask], lr_playoff_preds[mask]),
        'ELO':       brier_score_loss(y_playoff[mask], elo_preds_playoff[mask]),
    })

brier_by_season_df = pd.DataFrame(brier_by_season)

plt.figure(figsize=(12, 5))
for col, color in [('XGBoost', 'steelblue'), ('Logistic', 'darkorange'), ('ELO', 'green')]:
    plt.plot(brier_by_season_df['Season'], brier_by_season_df[col], marker='o', label=col, color=color)

plt.xlabel('Playoff Season')
plt.ylabel('Brier Score (lower = better)')
plt.title('Playoff Prediction Accuracy by Season')
plt.legend()
plt.tight_layout()
plt.savefig('output/brier_by_season.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Summary

| Model | Playoff Brier | Playoff AUC | Notes |
|---|---|---|---|
| ELO Baseline | — | — | Simple, interpretable floor |
| Logistic Regression | — | — | Good baseline, fully interpretable coefficients |
| XGBoost | — | — | Best performer; captures non-linear interactions |
| Neural Network | — | — | Similar to XGBoost; less interpretable |

*(Fill in values after running)*

**Key Findings:**
- The most predictive features are ELO rating differential and Net Rating differential
- Recent form (last 10 games) adds signal beyond season-long averages
- Away win percentage matters — road teams face extra pressure in the playoffs
- All models beat the ELO baseline on calibration (Brier score)